# 2 · Train the Transformer and watch it think

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rajdama/paper-to-pytorch/blob/main/notebooks/02_train_the_transformer.ipynb)

Steps 11–12 of [paper-to-pytorch](https://github.com/rajdama/paper-to-pytorch): train the full encoder-decoder
Transformer on a toy translation task (reverse a sequence of digits), decode with greedy
search and beam search, then look inside at what its cross-attention learned.

Runs in about a minute on a CPU; faster on a GPU (*Runtime → Change runtime type → T4 GPU*).

In [ ]:
# Get the code (Colab already has PyTorch and matplotlib installed)
import os
if not os.path.exists("step03_attention"):
    if not os.path.exists("paper-to-pytorch"):
        !git clone -q https://github.com/rajdama/paper-to-pytorch.git
    %cd paper-to-pytorch

import torch
from IPython.display import Image, display
print("PyTorch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU is fine for this notebook)")

## Train (label smoothing + the paper's warmup schedule)

In [ ]:
!python -m step11_training.train_reverse --steps 2000
display(Image("step11_training/training_curves.png"))

## Decode: greedy vs beam search

In [ ]:
!python -m step12_inference.demo
!python -m step12_inference.demo 3 1 4 1 5 9 2 6

## Look inside: cross-attention
To reverse a sequence, the decoder has to fetch the *last* source digit first. If the model
learned this, cross-attention lights up along the **anti-diagonal**.

In [ ]:
!python -m step12_inference.attention_map 2 7 1 8 2 8 1 8
display(Image("step12_inference/cross_attention.png"))

## Experiment: what if we remove positional encoding?
Attention by itself doesn't know token order. Without positions, the encoder can't tell
which digit came first, so reversing should become impossible. Let's check.

In [ ]:
import torch.nn as nn
from step10_full_transformer import Transformer
from step11_training import (FIRST_SYMBOL, PAD, LabelSmoothingLoss, make_optimizer,
                             reverse_task_batch)

device = "cuda" if torch.cuda.is_available() else "cpu"

def train(use_positions: bool, steps=1500):
    torch.manual_seed(0)
    model = Transformer(FIRST_SYMBOL + 10, FIRST_SYMBOL + 10, d_model=64, num_heads=4,
                        num_layers=2, d_ff=256, share_src_tgt_embeddings=True).to(device)
    if not use_positions:
        model.positional = nn.Identity()          # remove Step 02 entirely
    crit = LabelSmoothingLoss(0.1)
    opt, sched = make_optimizer(model, 64, warmup=200, factor=0.25)
    for step in range(steps):
        src, tgt = reverse_task_batch(64, device=device)
        logits = model(src, tgt[:, :-1])
        loss = crit(logits.reshape(-1, logits.size(-1)), tgt[:, 1:].reshape(-1))
        opt.zero_grad(); loss.backward(); opt.step(); sched.step()
    model.eval()
    with torch.no_grad():
        src, tgt = reverse_task_batch(500, device=device)
        logits = model(src, tgt[:, :-1])
    keep = tgt[:, 1:] != PAD
    return (logits.argmax(-1)[keep] == tgt[:, 1:][keep]).float().mean().item()

print(f"with positional encoding:    token accuracy {train(True):.1%}")
print(f"without positional encoding: token accuracy {train(False):.1%}")

**Questions to think about**
- Without positions, the accuracy isn't zero. Which tokens can it still get right, and why?
- In `step11_training/train_reverse.py`, try `--warmup 1`. Does the post-norm Transformer still train?

**Next:** [3 · TinyGPT](https://colab.research.google.com/github/rajdama/paper-to-pytorch/blob/main/notebooks/03_tinygpt.ipynb)